# Generate BB1 Phase Or Sine Amplitude Columns

This notebook demonstrates the repo-local waveform API. It builds a weighted BB1 phase table by default and also includes a simple sine amplitude waveform for comparison.

## 1. Setup

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the Hardware Capture repo root from a notebook or script working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        redstone_dir = candidate / "hardware_capture" / "rf" / "redstone"
        if (redstone_dir / "sequence.py").exists() and (redstone_dir / "waveform.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the Hardware Capture repo root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
EXAMPLES_DIR = REDSTONE_DIR / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)

import math
from fractions import Fraction
from hardware_capture.rf.redstone import BB1, Sine

## 2. Configure The Waveform

In [ ]:
OUTPUT_NAME = "phase"
OUTPUT_PRECISION = 3
TOTAL_PHASE_POINTS = 256
TARGET_ANGLE_DEG = 180.0
ROTATION_AXIS = "-X"


def axis_phase_deg(axis: str) -> float:
    normalized = axis.strip().upper()
    if normalized == "+X":
        return 0.0
    if normalized == "-X":
        return 180.0
    raise ValueError("ROTATION_AXIS must be '+X' or '-X'.")


phase_waveform = BB1(
    angle=TARGET_ANGLE_DEG,
    phase=axis_phase_deg(ROTATION_AXIS),
    split=True,
    repeats=1,
    phase_steps=1024,
)

amplitude_waveform = Sine(
    points=16,
    amp_min=0.0,
    amp_max=1.0,
    cycles=1.0,
    offset_deg=0.0,
)

phase_waveform.summary().to_dict()

## 3. Helper Functions For Weighted BB1 Tables

The BB1 phases are expanded according to pulse flip-angle weights so the output table has an exact requested point count.

In [ ]:
def bb1_segment_angles() -> list[float]:
    correction = [180.0, 360.0, 180.0]
    if getattr(phase_waveform, "correction_only", False):
        return correction
    if getattr(phase_waveform, "split", False):
        half_angle = phase_waveform.angle / 2.0
        return [half_angle, *correction, half_angle]
    return [phase_waveform.angle, *correction]


def ratio_string(values: list[float]) -> str:
    fractions = [
        Fraction(f"{value:.6f}".rstrip("0").rstrip(".")).limit_denominator(1000000)
        for value in values
    ]
    denominator = 1
    for fraction in fractions:
        denominator = math.lcm(denominator, fraction.denominator)
    integers = [
        fraction.numerator * (denominator // fraction.denominator)
        for fraction in fractions
    ]
    divisor = integers[0]
    for value in integers[1:]:
        divisor = math.gcd(divisor, value)
    return ":".join(str(value // divisor) for value in integers)


def weighted_phase_values(total_points: int):
    base_values = phase_waveform.phase_values()
    segment_angles = bb1_segment_angles()
    if base_values is None:
        raise ValueError("Phase waveform output is not available.")
    if len(base_values) != len(segment_angles):
        raise ValueError("Weighted BB1 export expects one phase per BB1 segment.")
    if total_points <= 0:
        raise ValueError("total_points must be greater than zero.")

    weight_total = sum(segment_angles)
    scale = total_points / weight_total
    counts = [int(round(weight * scale)) for weight in segment_angles]
    mismatch = total_points - sum(counts)
    if mismatch:
        center = len(counts) // 2
        counts[center] += mismatch
    if any(count <= 0 for count in counts):
        raise ValueError("total_points is too small for the requested BB1 weighting.")

    values = []
    for value, count in zip(base_values, counts, strict=True):
        values.extend([value] * count)
    if len(values) != total_points:
        raise ValueError("Weighted BB1 export did not produce the requested point count.")
    return values, counts, base_values, segment_angles


def weighted_phase_column(total_points: int, precision: int):
    values, counts, base_values, segment_angles = weighted_phase_values(total_points)
    lines = [f"{value:.{precision}f}".rstrip("0").rstrip(".") for value in values]
    return lines, counts, base_values, segment_angles

## 4. Generate The Phase Column

In [ ]:
phase_lines, counts, base_values, segment_angles = weighted_phase_column(
    total_points=TOTAL_PHASE_POINTS,
    precision=OUTPUT_PRECISION,
)

summary = {
    "axis": ROTATION_AXIS,
    "total_entries": TOTAL_PHASE_POINTS,
    "ratio": ratio_string(segment_angles),
    "portions": counts,
    "base_values": base_values,
    "segment_angles": segment_angles,
    "first_20": phase_lines[:20],
    "last_20": phase_lines[-20:],
}
summary

## 5. Optional Amplitude Column

In [ ]:
amplitude_lines = amplitude_waveform.amplitude_column(precision=OUTPUT_PRECISION)
{
    "amplitude_rows": len(amplitude_lines or []),
    "values": amplitude_lines,
}